In [1]:
from google import genai
from dotenv import load_dotenv
import os
import re
import pandas as pd

In [2]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

In [14]:
allowed_categories = [
    "Order Issue",
    "Delivery Issue",
    "Product Issue",
    "Return/Replacement",
    "Refund Issue",
    "Payment Issue",
    "Account Issue",
    "Cancellation",
    "General Inquiry"
]

allowed_priorities = [
    "Low",
    "Medium",
    "High"
]

In [15]:
def classify_complaint(complaint):

    prompt = f"""
    You are an e-commerce customer complaint classification system.

    Analyze the following customer complaint.

    Complaint:
    {complaint}

    Choose exactly one category:

    - Order Issue
    - Delivery Issue
    - Product Issue
    - Return/Replacement
    - Refund Issue
    - Payment Issue
    - Account Issue
    - Cancellation
    - General Inquiry

    Choose exactly one priority:

    - Low
    - Medium
    - High

    Return only:

    Category: <category>
    Priority: <priority>
    """

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

In [16]:
def parse_classification(result):

    category = None
    priority = None

    for line in result.splitlines():

        if line.startswith("Category:"):
            category = line.replace("Category:", "").strip()

        elif line.startswith("Priority:"):
            priority = line.replace("Priority:", "").strip()

    return category, priority

In [17]:
required_fields = [
    "Order ID",
    "Product",
    "Issue",
    "Customer Request",
    "Date"
]


def process_complaint(complaint):

    prompt = f"""
    You are an e-commerce customer support input processing system.

    Analyze the following customer complaint.

    Customer Complaint:
    {complaint}

    Extract:

    1. Order ID
    2. Product
    3. Issue
    4. Customer Request
    5. Date

    Rules:

    - Extract information only if it is present.
    - Do not invent or assume information.
    - If information is missing, write "Not provided".

    Return exactly:

    Order ID: <value>
    Product: <value>
    Issue: <value>
    Customer Request: <value>
    Date: <value>
    """

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

In [18]:
def parse_processed_information(result):

    extracted = {}

    for line in result.splitlines():

        for field in required_fields:

            if line.startswith(field + ":"):

                value = line.split(":", 1)[1].strip()
                extracted[field] = value

    return extracted

In [19]:
def determine_resolution(
    complaint,
    classification,
    processed_information
):

    prompt = f"""
    You are an e-commerce customer support resolution system.

    Customer Complaint:
    {complaint}

    Complaint Classification:
    {classification}

    Extracted Information:
    {processed_information}

    Determine the appropriate resolution.

    Rules:

    - Use only the information provided.
    - Do not invent order details.
    - Do not invent company policies.
    - Do not guarantee refunds or replacements.
    - Do not assume policy eligibility.
    - If policy verification is required, clearly mention it.

    Return exactly:

    Issue: <main issue>
    Resolution: <recommended resolution>
    Reason: <brief reason>
    """

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

In [20]:
resolution_fields = [
    "Issue",
    "Resolution",
    "Reason"
]


def validate_resolution_output(result):

    extracted_fields = {}

    for line in result.splitlines():

        for field in resolution_fields:

            if line.startswith(field + ":"):

                value = line.split(":", 1)[1].strip()
                extracted_fields[field] = value

    missing_fields = [
        field
        for field in resolution_fields
        if field not in extracted_fields
    ]

    empty_fields = [
        field
        for field, value in extracted_fields.items()
        if not value
    ]

    return {
        "extracted_fields": extracted_fields,
        "missing_fields": missing_fields,
        "empty_fields": empty_fields,
        "valid": len(missing_fields) == 0 and len(empty_fields) == 0
    }

In [21]:
evaluation_data = [
    
    {
        "complaint": "I ordered a black Nike shoe last week. Order ID is #12345, but I received the wrong size. I want to exchange it.",
        "expected_category": "Return/Replacement"
    },

    {
        "complaint": "My order was supposed to arrive three days ago but it still hasn't been delivered.",
        "expected_category": "Delivery Issue"
    },

    {
        "complaint": "The headphones I received are damaged and I want a replacement.",
        "expected_category": "Product Issue"
    },

    {
        "complaint": "I was charged twice for the same order.",
        "expected_category": "Payment Issue"
    },

    {
        "complaint": "I want my money back for the product I purchased.",
        "expected_category": "Refund Issue"
    },

    {
        "complaint": "Please cancel my order before it is shipped.",
        "expected_category": "Cancellation"
    },

    {
        "complaint": "I cannot log into my account even though my password is correct.",
        "expected_category": "Account Issue"
    },

    {
        "complaint": "The product I received is completely different from the one I ordered.",
        "expected_category": "Order Issue"
    },

    {
        "complaint": "I want to return the shoes I purchased.",
        "expected_category": "Return/Replacement"
    },

    {
        "complaint": "Can I change the delivery address for my order?",
        "expected_category": "General Inquiry"
    }
]

evaluation_df = pd.DataFrame(evaluation_data)

evaluation_df

,complaint,expected_category
0,I ordered a black Nike shoe last week. Order I...,Return/Replacement
1,My order was supposed to arrive three days ago...,Delivery Issue
2,The headphones I received are damaged and I wa...,Product Issue
3,I was charged twice for the same order.,Payment Issue
4,I want my money back for the product I purchased.,Refund Issue
5,Please cancel my order before it is shipped.,Cancellation
6,I cannot log into my account even though my pa...,Account Issue
7,The product I received is completely different...,Order Issue
8,I want to return the shoes I purchased.,Return/Replacement
9,Can I change the delivery address for my order?,General Inquiry


In [22]:
classification_results = []

for index, row in evaluation_df.iterrows():

    print(f"Processing complaint {index + 1}/{len(evaluation_df)}...")

    result = classify_complaint(row["complaint"])

    category, priority = parse_classification(result)

    classification_results.append({
        "complaint": row["complaint"],
        "expected_category": row["expected_category"],
        "predicted_category": category,
        "priority": priority
    })

print("\nClassification evaluation completed.")

Processing complaint 1/10...
Processing complaint 2/10...
Processing complaint 3/10...
Processing complaint 4/10...
Processing complaint 5/10...
Processing complaint 6/10...
Processing complaint 7/10...
Processing complaint 8/10...
Processing complaint 9/10...
Processing complaint 10/10...

Classification evaluation completed.


In [23]:
classification_df = pd.DataFrame(classification_results)

classification_df

,complaint,expected_category,predicted_category,priority
0,I ordered a black Nike shoe last week. Order I...,Return/Replacement,Return/Replacement,Medium
1,My order was supposed to arrive three days ago...,Delivery Issue,Delivery Issue,Medium
2,The headphones I received are damaged and I wa...,Product Issue,Product Issue,Medium
3,I was charged twice for the same order.,Payment Issue,Payment Issue,High
4,I want my money back for the product I purchased.,Refund Issue,Refund Issue,Medium
5,Please cancel my order before it is shipped.,Cancellation,Cancellation,High
6,I cannot log into my account even though my pa...,Account Issue,Account Issue,Medium
7,The product I received is completely different...,Order Issue,Order Issue,High
8,I want to return the shoes I purchased.,Return/Replacement,Return/Replacement,Low
9,Can I change the delivery address for my order?,General Inquiry,Delivery Issue,Medium


In [25]:
import time
extraction_results = []

for index, row in evaluation_df.iterrows():

    print(f"Extracting complaint {index + 1}/{len(evaluation_df)}...")

    try:
        result = process_complaint(row["complaint"])

        extracted = parse_processed_information(result)

        all_fields_present = all(
            field in extracted
            for field in required_fields
        )

        extraction_results.append({
            "complaint": row["complaint"],
            "all_fields_present": all_fields_present,
            "extracted_information": extracted
        })

    except Exception as e:

        print(f"Error on complaint {index + 1}: {e}")

        extraction_results.append({
            "complaint": row["complaint"],
            "all_fields_present": False,
            "extracted_information": {}
        })

    time.sleep(2)

print("\nExtraction evaluation completed.")

Extracting complaint 1/10...
Extracting complaint 2/10...
Extracting complaint 3/10...
Extracting complaint 4/10...
Extracting complaint 5/10...
Extracting complaint 6/10...
Extracting complaint 7/10...
Extracting complaint 8/10...
Extracting complaint 9/10...
Extracting complaint 10/10...

Extraction evaluation completed.


In [26]:
extraction_df = pd.DataFrame(extraction_results)

extraction_df

,complaint,all_fields_present,extracted_information
0,I ordered a black Nike shoe last week. Order I...,True,"{'Order ID': '#12345', 'Product': 'black Nike ..."
1,My order was supposed to arrive three days ago...,True,"{'Order ID': 'Not provided', 'Product': 'Not p..."
2,The headphones I received are damaged and I wa...,True,"{'Order ID': 'Not provided', 'Product': 'headp..."
3,I was charged twice for the same order.,True,"{'Order ID': 'Not provided', 'Product': 'Not p..."
4,I want my money back for the product I purchased.,True,"{'Order ID': 'Not provided', 'Product': 'Not p..."
5,Please cancel my order before it is shipped.,True,"{'Order ID': 'Not provided', 'Product': 'Not p..."
6,I cannot log into my account even though my pa...,True,"{'Order ID': 'Not provided', 'Product': 'Not p..."
7,The product I received is completely different...,True,"{'Order ID': 'Not provided', 'Product': 'Not p..."
8,I want to return the shoes I purchased.,True,"{'Order ID': 'Not provided', 'Product': 'shoes..."
9,Can I change the delivery address for my order?,True,"{'Order ID': 'Not provided', 'Product': 'Not p..."


In [34]:
extraction_accuracy = (
    extraction_df["all_fields_present"].mean() * 100
)

print(f"Extraction Structure Score: {extraction_accuracy:.2f}%")

Extraction Structure Score: 100.00%


In [30]:
import time

resolution_results = []

for index, row in evaluation_df.iterrows():

    print(f"\nGenerating resolution {index + 1}/{len(evaluation_df)}...")

    try:
        # Step 1: Classification
        print("  → Classification...")
        classification = classify_complaint(row["complaint"])
        time.sleep(2)

        # Step 2: Process information
        print("  → Processing information...")
        processed_information = process_complaint(row["complaint"])
        time.sleep(2)

        # Step 3: Generate resolution
        print("  → Generating resolution...")
        resolution = determine_resolution(
            row["complaint"],
            classification,
            processed_information
        )
        time.sleep(2)

        # Step 4: Validate
        validation = validate_resolution_output(resolution)

        resolution_results.append({
            "complaint": row["complaint"],
            "resolution": resolution,
            "valid": validation["valid"]
        })

        print("  ✓ Completed")

    except Exception as e:

        print(f"  ✗ Error: {e}")

        resolution_results.append({
            "complaint": row["complaint"],
            "resolution": "API Error",
            "valid": False
        })

print("\nResolution evaluation completed.")


Generating resolution 1/10...
  → Classification...
  → Processing information...
  → Generating resolution...
  ✓ Completed

Generating resolution 2/10...
  → Classification...
  → Processing information...
  → Generating resolution...
  ✓ Completed

Generating resolution 3/10...
  → Classification...
  → Processing information...
  → Generating resolution...
  ✓ Completed

Generating resolution 4/10...
  → Classification...
  → Processing information...
  → Generating resolution...
  ✓ Completed

Generating resolution 5/10...
  → Classification...
  → Processing information...
  → Generating resolution...
  ✓ Completed

Generating resolution 6/10...
  → Classification...
  → Processing information...
  → Generating resolution...
  ✓ Completed

Generating resolution 7/10...
  → Classification...
  → Processing information...
  → Generating resolution...
  ✓ Completed

Generating resolution 8/10...
  → Classification...
  → Processing information...
  → Generating resolution...
  ✓ Co

In [31]:
resolution_df = pd.DataFrame(resolution_results)

resolution_df

,complaint,resolution,valid
0,I ordered a black Nike shoe last week. Order I...,Issue: Received the wrong size for the black N...,True
1,My order was supposed to arrive three days ago...,Issue: Late delivery / order delayed by three ...,True
2,The headphones I received are damaged and I wa...,Issue: Damaged headphones\nResolution: Request...,True
3,I was charged twice for the same order.,Issue: Charged twice for the same order\nResol...,True
4,I want my money back for the product I purchased.,Issue: Request for a refund without specific o...,True
5,Please cancel my order before it is shipped.,Issue: Customer wants to cancel their order be...,True
6,I cannot log into my account even though my pa...,Issue: Cannot log into account even though pas...,True
7,The product I received is completely different...,Issue: The product received is completely diff...,True
8,I want to return the shoes I purchased.,Issue: Return the shoes\nResolution: Request t...,True
9,Can I change the delivery address for my order?,API Error,False


In [37]:
classification_df["category_correct"] = (
    classification_df["expected_category"]
    ==
    classification_df["predicted_category"]
)

classification_accuracy = (
    classification_df["category_correct"].mean() * 100
)

print(f"Classification Accuracy: {classification_accuracy:.2f}%")

Classification Accuracy: 90.00%


In [38]:
resolution_score = (
    resolution_df["valid"].mean() * 100
)

print(f"Resolution Validity Score: {resolution_score:.2f}%")

Resolution Validity Score: 90.00%


In [39]:
evaluation_summary = pd.DataFrame({
    "Metric": [
        "Classification Accuracy",
        "Extraction Structure Score",
        "Resolution Validity Score"
    ],
    
    "Score": [
        classification_accuracy,
        extraction_accuracy,
        resolution_score
    ]
})

evaluation_summary

,Metric,Score
0,Classification Accuracy,90.0
1,Extraction Structure Score,100.0
2,Resolution Validity Score,90.0


In [41]:
overall_score = (
    classification_accuracy +
    extraction_accuracy +
    resolution_score
) / 3

print(f"Overall Evaluation Score: {overall_score:.2f}%")

Overall Evaluation Score: 93.33%


In [45]:

print("AI CUSTOMER COMPLAINT RESOLUTION ASSISTANT")
print("EVALUATION REPORT")


print(f"\nNumber of Test Complaints : {len(evaluation_df)}")

print(
    f"Classification Accuracy   : "
    f"{classification_accuracy:.2f}%"
)

print(
    f"Extraction Structure Score: "
    f"{extraction_accuracy:.2f}%"
)

print(
    f"Resolution Validity Score : "
    f"{resolution_score:.2f}%"
)

print(
    f"Overall Evaluation Score  : "
    f"{overall_score:.2f}%"
)



AI CUSTOMER COMPLAINT RESOLUTION ASSISTANT
EVALUATION REPORT

Number of Test Complaints : 10
Classification Accuracy   : 90.00%
Extraction Structure Score: 100.00%
Resolution Validity Score : 90.00%
Overall Evaluation Score  : 93.33%
